# Utilities

In [ ]:
gather_data = False
save_data = True

## Libraries

In [2]:
import pandas as pd
import json
import sys
sys.path.append('..')
from xploreapi.xploreapi import XPLORE
import matplotlib.pyplot as plt

## Functions

In [3]:
# def AnalisisNan(data):
#   data_nan = 100 - data.isna().sum(axis=0) / len(data) * 100
#   data_nan = data_nan.sort_values(ascending=False)
#   data_nan = data_nan.round(2)
#   colors = sns.color_palette("viridis", len(data_nan))
#   plt.figure(figsize=(6, int(0.3 * len(data_nan))))
#   plt.barh(data_nan.index, data_nan, color=colors)
#   for idx, val in enumerate(data_nan):
#     plt.text(val, idx, f'{val}%')
#   plt.title('Porcentaje de datos disponibles por columna')
#   plt.ylabel('Columna')
#   plt.xlabel('% de datos disponibles (no NaN)')
#   plt.show()

# For index_terms treatment, list_flatten and dict_flatten are defined.

def list_flatten(my_list):

    result = []

    # Iterate over my_list
    for element in my_list:

        # If it is a list, unpack and append to result
        if isinstance(element, list):
            result.extend(list_flatten(element))

        # If it is a string
        elif isinstance(element, str):
            element = element.strip() # Remove start/end spaces
            result.append(element)    # Append to result
        
        # Else raise error
        else:
            raise TypeError('Element is not list or string.')

    return result

def dict_flatten(my_dict):
    
    result = []

    # Iterate over my_dict
    for key, val in my_dict.items():

        # If it is a dictionary, unpack and append to result
        if isinstance(val, dict):
            result.extend(dict_flatten(val))
        
        # If it is a list, unpack and append to result
        elif isinstance(val, list):
            result.extend(val)        
        
        # Else raise error
        else:
            raise TypeError('Element is not dictionary or list')
    
    return result

# Gather data

## IEEEXplore API

In [4]:
if gather_data:

    import private_constants

    # Cargar la clave API desde el archivo de constantes privadas
    api_key = private_constants.API_KEY_IEEEXplore

    # Inicializar el cliente de la API
    client = XPLORE(api_key)

    # Buscar en metadatos, abstract y texto completo
    client.queryText("modular multilevel converter")

    # Número máximo de documentos que deseas descargar
    max_docs_to_download = 15000

    # Configurar número máximo por solicitud
    batch_size = 200
    client.maximumResults(batch_size)

    # Obtener el número total de documentos disponibles (limitado por max_docs_to_download)
    response = client.callAPI()
    data = json.loads(response)
    total_docs = min(data.get("total_records", 0), max_docs_to_download)

    print(f"# of documents to download: {total_docs}")

    # Descargar todos los documentos por lotes
    all_records = []
    for start in range(1, total_docs + 1, batch_size):
        client.startingResult(start)

        response = client.callAPI()
        data = json.loads(response)
        
        batch_records = data.get("articles", [])
        all_records.extend(batch_records)
        
        print(f"Downloaded: {len(all_records)} of {total_docs}")

        if len(batch_records) < batch_size:
            break

    print(f"Total documents downloaded: {len(all_records)}")

    data = pd.DataFrame(all_records)

    if save_data:
        data.to_csv('../data/input/api/data_raw.csv', index=False)

# of documents to download: 11527
Downloaded: 200 of 11527
Downloaded: 400 of 11527
Downloaded: 600 of 11527
Downloaded: 800 of 11527
Downloaded: 1000 of 11527
Downloaded: 1200 of 11527
Downloaded: 1400 of 11527
Downloaded: 1600 of 11527
Downloaded: 1800 of 11527
Downloaded: 2000 of 11527
Downloaded: 2200 of 11527
Downloaded: 2400 of 11527
Downloaded: 2600 of 11527
Downloaded: 2800 of 11527
Downloaded: 3000 of 11527
Downloaded: 3200 of 11527
Downloaded: 3400 of 11527
Downloaded: 3600 of 11527
Downloaded: 3800 of 11527
Downloaded: 4000 of 11527
Downloaded: 4200 of 11527
Downloaded: 4400 of 11527
Downloaded: 4600 of 11527
Downloaded: 4800 of 11527
Downloaded: 5000 of 11527
Downloaded: 5200 of 11527
Downloaded: 5400 of 11527
Downloaded: 5600 of 11527
Downloaded: 5800 of 11527
Downloaded: 6000 of 11527
Downloaded: 6200 of 11527
Downloaded: 6400 of 11527
Downloaded: 6600 of 11527
Downloaded: 6800 of 11527
Downloaded: 7000 of 11527
Downloaded: 7200 of 11527
Downloaded: 7400 of 11527
Download

In [5]:
data = pd.read_csv('../data/input/api/data_raw.csv')
data.reset_index(inplace=True, drop=True)
print(len(data))
data.drop_duplicates(inplace=True)
print(len(data))

11527
11527


In [6]:
print(data.shape)
display(data.head())
display(data.isna().sum().sort_values())

(11527, 36)


,doi,title,publisher,issn,rank,volume,authors,access_type,content_type,abstract,...,license,issue,isbn,conference_location,conference_dates,isbn_formats,partnum,supplementGroup,standard_number,standard_status
0,10.1109/OJPEL.2024.3387076,$\beta$-Ga2O3 in Power Electronics Converters:...,IEEE,2644-1314,1,5,{'authors': [{'affiliation': 'Electrical Energ...,CCBY - IEEE is not the copyright holder of thi...,Journals,"In this work, the possibility of using differe...",...,CCBY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1109/TPEL.2017.2666847,(2N+1) Selective Harmonic Elimination-PWM for ...,IEEE,1941-0107,2,33,{'authors': [{'affiliation': 'Department of El...,LOCKED,Journals,The performance of modular multilevel converte...,...,IEEE,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10.1109/ICASSP.2019.8682194,1-D Convolutional Neural Networks for Signal P...,IEEE,1520-6149,3,NaN,{'authors': [{'affiliation': 'Department of El...,LOCKED,Conferences,1D Convolutional Neural Networks (CNNs) have r...,...,IEEE,NaN,978-1-4799-8132-8,"Brighton, UK",12-17 May 2019,{'isbns': [{'format': 'Print on Demand(PoD) IS...,NaN,NaN,NaN,NaN
3,NaN,1-MW Full-Bridge MMC for High-Current Low-Volt...,VDE,NaN,4,NaN,"{'authors': [{'full_name': 'Roland Unruh', 'au...",LOCKED,Conferences,A full-bridge modular multilevel converter (MM...,...,NaN,NaN,978-3-8007-5245-4,Germany,7-8 July 2020,"{'isbns': [{'format': 'Print ISBN', 'value': '...",NaN,NaN,NaN,NaN
4,10.1109/APEC.2019.8721944,1-kV Input 1-MHz GaN Modular Multilevel LLC Co...,IEEE,1048-2334,5,NaN,{'authors': [{'affiliation': 'Aero-Power Sci-T...,LOCKED,Conferences,A modular multilevel LLC converter suitable fo...,...,IEEE,NaN,978-1-5386-8331-6,"Anaheim, CA, USA",17-21 March 2019,{'isbns': [{'format': 'Print on Demand(PoD) IS...,NaN,NaN,NaN,NaN


title                      0
access_type                0
authors                    0
rank                       0
html_url                   0
publication_title          0
article_number             0
content_type               0
insert_date                0
citing_paper_count         0
download_count             0
citing_patent_count        0
publication_number         4
pdf_url                    4
publisher                  4
start_page                 4
publication_year           4
abstract                   6
end_page                  10
abstract_url              19
is_number                 55
publication_date         187
doi                      292
index_terms             1789
license                 1813
isbn                    3608
isbn_formats            3608
conference_location     3679
conference_dates        3679
issn                    4931
volume                  7500
issue                   8350
partnum                11108
standard_number        11505
standard_statu

In [7]:
print(data['insert_date'].isna().sum())
data['insert_date']

0


0        20240410
1        20170209
2        20190417
3        20200826
4        20190527
           ...   
11522    20201013
11523    20130121
11524    20190822
11525    20200721
11526    20230531
Name: insert_date, Length: 11527, dtype: int64

In [8]:
print(data['publication_year'].isna().sum())
data['publication_year']

4


0        2024.0
1        2018.0
2        2019.0
3        2020.0
4        2019.0
          ...  
11522    2021.0
11523    2012.0
11524    2019.0
11525    2020.0
11526    2023.0
Name: publication_year, Length: 11527, dtype: float64

In [9]:
mask = data['publication_year'].isna()
data.loc[mask, 'publication_year']

6528   NaN
6530   NaN
6532   NaN
6543   NaN
Name: publication_year, dtype: float64

In [10]:
display(data.loc[mask, 'insert_date'] / 10000)
display(round(data.loc[mask, 'insert_date'] / 10000))

6528    2019.0425
6530    2020.1224
6532    2022.0809
6543    2019.1125
Name: insert_date, dtype: float64

6528    2019.0
6530    2020.0
6532    2022.0
6543    2019.0
Name: insert_date, dtype: float64

In [11]:
data.loc[mask, 'publication_year'] = round(data.loc[mask, 'insert_date'] / 10000)
data.loc[mask, 'publication_year']

6528    2019.0
6530    2020.0
6532    2022.0
6543    2019.0
Name: publication_year, dtype: float64

In [12]:
print(data['publication_year'].isna().sum())

0


In [13]:
data.columns

Index(['doi', 'title', 'publisher', 'issn', 'rank', 'volume', 'authors',
       'access_type', 'content_type', 'abstract', 'article_number', 'pdf_url',
       'html_url', 'abstract_url', 'publication_title', 'publication_number',
       'is_number', 'publication_year', 'publication_date', 'start_page',
       'end_page', 'citing_paper_count', 'citing_patent_count',
       'download_count', 'insert_date', 'index_terms', 'license', 'issue',
       'isbn', 'conference_location', 'conference_dates', 'isbn_formats',
       'partnum', 'supplementGroup', 'standard_number', 'standard_status'],
      dtype='object')

In [14]:
# The useful columns are:
cols2keep = [
    'title',
    'publisher',
    'authors',
    'content_type',
    'abstract',
    'html_url',
    'publication_title',
    'publication_year',
    'citing_paper_count',
    'citing_patent_count',
    'download_count',
    'index_terms'
]

data = data[cols2keep].copy()

print(data.shape)
display(data.dtypes)
display(data.isna().sum().sort_values())

(11527, 12)


title                   object
publisher               object
authors                 object
content_type            object
abstract                object
html_url                object
publication_title       object
publication_year       float64
citing_paper_count       int64
citing_patent_count      int64
download_count           int64
index_terms             object
dtype: object

title                     0
authors                   0
content_type              0
html_url                  0
publication_year          0
publication_title         0
citing_patent_count       0
citing_paper_count        0
download_count            0
publisher                 4
abstract                  6
index_terms            1789
dtype: int64

In [15]:
data.reset_index(inplace=True, drop=True)

# Missing values

## publisher

In [16]:
mask = data['publisher'].isna()
data[mask]

,title,publisher,authors,content_type,abstract,html_url,publication_title,publication_year,citing_paper_count,citing_patent_count,download_count,index_terms
6528,IEEE Draft Guide for Technology of Unified Pow...,NaN,{'authors': []},Standards,An approach to preparing a functional specific...,https://ieeexplore.ieee.org/document/STDUD23659/,IEEE Draft Guide for Technology of Unified Pow...,2019.0,0,0,0,NaN
6530,IEEE Draft Guide for Technology of Unified Pow...,NaN,{'authors': []},Standards,This guide specifies glossary and definition o...,https://ieeexplore.ieee.org/document/STDUD24558/,IEEE Draft Guide for Technology of Unified Pow...,2020.0,0,0,0,NaN
6532,IEEE Draft Guide for Technology of Unified Pow...,NaN,{'authors': []},Standards,The application of control and protection on a...,https://ieeexplore.ieee.org/document/STDUD25567/,IEEE Draft Guide for Technology of Unified Pow...,2022.0,0,0,0,NaN
6543,IEEE Guide for Technology of Unified Power Flo...,NaN,{'authors': []},Standards,An approach to preparing a functional specific...,https://ieeexplore.ieee.org/document/STD23882/,IEEE Guide for Technology of Unified Power Flo...,2019.0,0,0,0,NaN


In [17]:
data.dropna(subset=['publisher'], inplace=True)

## abstract

In [18]:
mask = data['abstract'].isna()
data[mask]

,title,publisher,authors,content_type,abstract,html_url,publication_title,publication_year,citing_paper_count,citing_patent_count,download_count,index_terms
5961,Front Matter,IEEE,{'authors': [{'full_name': 'Kamran Sharifabadi...,Books,NaN,https://ieeexplore.ieee.org/document/7656763/,"Design, Control, and Application of Modular Mu...",2016.0,2,0,845,NaN
6550,IEEE Press Series on Power and Energy Systems,IEEE,"{'authors': [{'full_name': 'Fujin Deng', 'auth...",Books,NaN,https://ieeexplore.ieee.org/document/10097772/,"Modular Multilevel Converters: Control, Fault ...",2023.0,0,0,0,NaN
6866,Index,IEEE,{'authors': [{'full_name': 'Kamran Sharifabadi...,Books,NaN,https://ieeexplore.ieee.org/document/7656798/,"Design, Control, and Application of Modular Mu...",2016.0,0,0,720,NaN
6867,Index,IEEE,"{'authors': [{'full_name': 'Fujin Deng', 'auth...",Books,NaN,https://ieeexplore.ieee.org/document/10097771/,"Modular Multilevel Converters: Control, Fault ...",2023.0,0,0,0,NaN
9139,Platform for Power,IET,"{'authors': [{'affiliation': 'Semikron, UK', '...",Magazines,NaN,https://ieeexplore.ieee.org/document/1605531/,Power Engineer,2005.0,0,0,9,NaN
10825,Table of Contents,IEEE,{'authors': []},Conferences,NaN,https://ieeexplore.ieee.org/document/10261900/,2023 IEEE Workshop on Wide Bandgap Power Devic...,2023.0,0,0,41,NaN


In [19]:
data.dropna(subset=['abstract'], inplace=True)

## index_terms

Missing values handling and index terms flattening

In [20]:
data['index_terms']

0        {'ieee_terms': {'terms': ['Schottky diodes', '...
1        {'ieee_terms': {'terms': ['Switches', 'Harmoni...
2                                                      NaN
3                                                      NaN
4        {'ieee_terms': {'terms': ['Silicon carbide', '...
                               ...                        
11522    {'ieee_terms': {'terms': ['Impedance', 'Protec...
11523                                                  NaN
11524    {'ieee_terms': {'terms': ['Topology', 'Bridge ...
11525                                                  NaN
11526    {'ieee_terms': {'terms': ['Pulse generation', ...
Name: index_terms, Length: 11517, dtype: object

In [21]:
fill_dict = "{'ieee_terms': {'terms': ['']}, 'author_terms': {'terms': ['']}, 'dynamic_index_terms': {'terms': ['']}}"
data['index_terms'] = data['index_terms'].fillna(fill_dict).copy()

In [22]:
# Sample index_terms value
sample = data.iloc[0]['index_terms']

In [23]:
# The solution is to append all terms to a flat list
dic_terms = eval(sample)
dic_terms

{'ieee_terms': {'terms': ['Schottky diodes',
   'Costs',
   'Performance evaluation',
   'Diamonds',
   'Silicon carbide',
   'Power electronics',
   'Thermal conductivity']},
 'author_terms': {'terms': ['  $\\beta$  -Ga  $_2$  0  $_3$  ',
   'diamond',
   'gallium oxide',
   'HVDC converters',
   'ultra-wide-bandgap']},
 'dynamic_index_terms': {'terms': ['Power Electronics',
   'Power Electronic Converters',
   'High Power',
   'Temperature Rise',
   'Incumbent',
   'Re-election',
   'Device Applications',
   'Power Loss',
   'Thermal Resistance',
   'Power Conversion',
   'Computer-aided Design',
   'Power Devices',
   'Power Transistors',
   'Power Semiconductor Devices',
   'Semiconductor Devices',
   'Converter Operation',
   'Performance Of Converter',
   'Power Semiconductor',
   'Modular Multilevel Converter',
   'On-state Resistance',
   'Blocking Voltage',
   'High Voltage',
   'Higher Tendency',
   'Thermal Conductivity',
   'Heat Conduction',
   'Power System',
   'Si Devic

In [24]:
# Flatten dictionary
dict_flatten(dic_terms)

['Schottky diodes',
 'Costs',
 'Performance evaluation',
 'Diamonds',
 'Silicon carbide',
 'Power electronics',
 'Thermal conductivity',
 '  $\\beta$  -Ga  $_2$  0  $_3$  ',
 'diamond',
 'gallium oxide',
 'HVDC converters',
 'ultra-wide-bandgap',
 'Power Electronics',
 'Power Electronic Converters',
 'High Power',
 'Temperature Rise',
 'Incumbent',
 'Re-election',
 'Device Applications',
 'Power Loss',
 'Thermal Resistance',
 'Power Conversion',
 'Computer-aided Design',
 'Power Devices',
 'Power Transistors',
 'Power Semiconductor Devices',
 'Semiconductor Devices',
 'Converter Operation',
 'Performance Of Converter',
 'Power Semiconductor',
 'Modular Multilevel Converter',
 'On-state Resistance',
 'Blocking Voltage',
 'High Voltage',
 'Higher Tendency',
 'Thermal Conductivity',
 'Heat Conduction',
 'Power System',
 'Si Devices',
 'Power Sector',
 'Rated Power',
 'Converter Topology',
 'Switching Loss',
 'Schottky Diode',
 'Schottky Barrier Diodes',
 'Submodule',
 'Thermal Management'

In [25]:
# Flatten list
list_flatten(dict_flatten(dic_terms))

['Schottky diodes',
 'Costs',
 'Performance evaluation',
 'Diamonds',
 'Silicon carbide',
 'Power electronics',
 'Thermal conductivity',
 '$\\beta$  -Ga  $_2$  0  $_3$',
 'diamond',
 'gallium oxide',
 'HVDC converters',
 'ultra-wide-bandgap',
 'Power Electronics',
 'Power Electronic Converters',
 'High Power',
 'Temperature Rise',
 'Incumbent',
 'Re-election',
 'Device Applications',
 'Power Loss',
 'Thermal Resistance',
 'Power Conversion',
 'Computer-aided Design',
 'Power Devices',
 'Power Transistors',
 'Power Semiconductor Devices',
 'Semiconductor Devices',
 'Converter Operation',
 'Performance Of Converter',
 'Power Semiconductor',
 'Modular Multilevel Converter',
 'On-state Resistance',
 'Blocking Voltage',
 'High Voltage',
 'Higher Tendency',
 'Thermal Conductivity',
 'Heat Conduction',
 'Power System',
 'Si Devices',
 'Power Sector',
 'Rated Power',
 'Converter Topology',
 'Switching Loss',
 'Schottky Diode',
 'Schottky Barrier Diodes',
 'Submodule',
 'Thermal Management',
 '

In [26]:
# Flatten all data index_terms
data['index_terms'] = data['index_terms'].apply(eval).apply(dict_flatten).apply(list_flatten)
data['index_terms']

0        [Schottky diodes, Costs, Performance evaluatio...
1        [Switches, Harmonic analysis, Pulse width modu...
2                                                   [, , ]
3                                                   [, , ]
4        [Silicon carbide, Inductors, HEMTs, MODFETs, S...
                               ...                        
11522    [Impedance, Protective relaying, Resistance, T...
11523                                               [, , ]
11524    [Topology, Bridge circuits, Insulation, Load m...
11525                                               [, , ]
11526    [Pulse generation, Systematics, Protocols, Tra...
Name: index_terms, Length: 11517, dtype: object

In [27]:
# Concatenate all index_terms for every row
data['index_terms'] = data['index_terms'].apply(lambda x: ' '.join(x))
data['index_terms']

0        Schottky diodes Costs Performance evaluation D...
1        Switches Harmonic analysis Pulse width modulat...
2                                                         
3                                                         
4        Silicon carbide Inductors HEMTs MODFETs Stress...
                               ...                        
11522    Impedance Protective relaying Resistance Trans...
11523                                                     
11524    Topology Bridge circuits Insulation Load model...
11525                                                     
11526    Pulse generation Systematics Protocols Transcr...
Name: index_terms, Length: 11517, dtype: object

In [28]:
mask = data['index_terms'].apply(len) == 2
data.loc[mask, 'index_terms'] = 'missing'

# Save data

In [29]:
data.isna().sum().sort_values()

title                  0
publisher              0
authors                0
content_type           0
abstract               0
html_url               0
publication_title      0
publication_year       0
citing_paper_count     0
citing_patent_count    0
download_count         0
index_terms            0
dtype: int64

In [30]:
data.shape

(11517, 12)

In [31]:
if save_data:
    data.to_csv('../data/input/api/data.csv', index=False)